# Frozen retrained-probe steering on seeds 100–103

This is an **evaluation-only** notebook. It loads the completed probe weights
and the signs, nonnegative combination weights, layers, and 0.045 strength
selected in the preceding experiment. No probe regression, classifier,
direction sign, direction weight, or steering-strength fit occurs here.

The preceding run did not export its two activation-norm scalars. This notebook
reconstructs those scalars exactly by replaying the same four logged source
arms and the same controlled counterfactual examples, then forwarding their
action-conditioned prompts through the unchanged model. It checks the exact
source replay and training-example counts. Only the two fixed scalars are
recovered; no probe weights or decisions are learned from these examples.

Four paired seeds, 100–103, then compare baseline, equal signed steering,
and learned-weight signed steering for two simulated hours, 400 bee decisions
per arm. Each bee still receives only its local observation and feasible menu.
The researcher-only 14-component hive state is snapshotted after every
decision. The notebook saves all final vectors, a timestamped step-change
transcript, paired score differences, and overlaid trajectories.

Seeds 100–103 were used in earlier experiments with other probe directions.
They were not used to fit the retrained probe or its new direction settings,
but they are not wholly untouched study seeds. Four paired seeds improve
precision over the preceding two-seed result; they do not establish a broad
biological or simulator-general effect.

In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo "transformers>=5.0.0" "trl>=0.27.0" datasets accelerate peft scikit-learn

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import copy, csv, gzip, json, math, sys, time, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from unsloth import FastLanguageModel

INPUT_ROOT=Path('/kaggle/input')
runtime_archive=next(INPUT_ROOT.rglob('beehive_qwen35_runtime_bundle.zip'),None)
menu_module=next(INPUT_ROOT.rglob('bee_menu_qwen_policy.py'))
PROBE_ROOT=next(INPUT_ROOT.rglob('new_probe_layer_14.npz')).parent
probe_files={layer:PROBE_ROOT/f'new_probe_layer_{layer}.npz' for layer in (13,14)}
PRIOR_REPORT=json.loads((PROBE_ROOT/'experiment_report.json').read_text())
TRAINED_ROOT=next(INPUT_ROOT.rglob('retrained_probe_layer_13.npz')).parent
TRAINED_REPORT=json.loads((TRAINED_ROOT/'retrained_probe_report.json').read_text())
adapter_files=[p for p in INPUT_ROOT.rglob('adapter_model.safetensors')
               if 'menu' in str(p.parent).lower()]
if not adapter_files: raise FileNotFoundError('Attach the private continued menu-LoRA adapter dataset')
ADAPTER_DIR=adapter_files[0].parent
RUN_ROOT=Path('/kaggle/working')/('bee_retrained_holdout_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN_ROOT.mkdir(parents=True,exist_ok=False)
if runtime_archive is not None:
    RUNTIME_DIR=RUN_ROOT/'runtime'
    with zipfile.ZipFile(runtime_archive) as z: z.extractall(RUNTIME_DIR)
else:
    RUNTIME_DIR=next(INPUT_ROOT.rglob('beehive_sim.py')).parent
sys.path[:0]=[str(menu_module.parent),str(RUNTIME_DIR)]
from beehive_sim import Config, World, Observer, Brood, Simulation, VECTOR_NAMES, weighted_research_score, validate_action
from bee_menu_qwen_policy import BeeMenuQwenPolicy, MENU_SYSTEM_PROMPT
from bee_feasibility import feasible_actions
SOURCE_META=json.loads(next(INPUT_ROOT.rglob('probe_source_metadata.json')).read_text())
SOURCE_CONFIG=dict(SOURCE_META['calibration']['config'])
assert len(VECTOR_NAMES)==14
print('Adapter',ADAPTER_DIR,'probe files',probe_files,'output',RUN_ROOT)

In [ ]:
SMOKE=False
BATCH_SIZE=16
MAX_NEW_TOKENS=160
FINAL_SEEDS=(100,) if SMOKE else (100,101,102,103)
HOURS=0.10 if SMOKE else 2.0
SELECTED_LAYERS=tuple(TRAINED_REPORT['selected_layers'])
assert SELECTED_LAYERS==(13,14)
assert TRAINED_REPORT['final_strength']=={
    'new_equal_combination':0.045,'new_learned_combination':0.045}
FRACTION=0.045
POSITION='all'

model,processor=FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),max_seq_length=4096,dtype=None,
    load_in_4bit=False,load_in_16bit=True,full_finetuning=False)
FastLanguageModel.for_inference(model)
model.eval()

def decoder_candidates(model):
    found=[]
    for name,module in model.named_modules():
        if not isinstance(module,nn.ModuleList) or len(module)<8:continue
        if not any(hasattr(module[0],field) for field in
                   ('self_attn','linear_attn','mlp')):continue
        score=len(module)+100*('language_model' in name)+50*('text' in name)
        score-=100*any(word in name for word in ('visual','vision','image'))
        found.append((score,name,module))
    return sorted(found,reverse=True)
_,LAYER_PATH,DECODER_LAYERS=decoder_candidates(model)[0]
assert len(DECODER_LAYERS)>max(SELECTED_LAYERS)

def normalized(vector):
    vector=np.asarray(vector,dtype=np.float32)
    size=float(np.linalg.norm(vector))
    assert np.isfinite(size) and size>1e-10
    return vector/size

ACTIVE=tuple(TRAINED_REPORT['representatives'])
W={layer:np.load(TRAINED_ROOT/f'retrained_probe_layer_{layer}.npz')['W']
   .astype(np.float32) for layer in SELECTED_LAYERS}
U={layer:W[layer][:,ACTIVE]/np.linalg.norm(W[layer][:,ACTIVE],axis=0)
   for layer in SELECTED_LAYERS}
assert len(ACTIVE)==8
activation_norms={}
print('Loaded frozen probe layers',SELECTED_LAYERS,'active',ACTIVE,
      'adapter',ADAPTER_DIR,'trained output',TRAINED_ROOT,'smoke',SMOKE)

## 1. Reconstruct the fixed activation norms without probe fitting

Source seeds 90 and 91 are replayed exactly, with baseline and old steered
worlds. The same targeted and controlled cases are constructed as in the
training notebook. In full mode the notebook asserts the prior count of
2,623 examples and every example-kind count. Neutral response activations
are pooled only at layers 13 and 14. Their median norms across the full
training examples are the two scalars used by the preceding intervention.
Smoke mode uses a small subset to check the code path; only full mode
reconstructs the final scalars.

In [ ]:
def config_for(seed,hours):
    data=dict(SOURCE_CONFIG)
    data.update(seed=int(seed),duration_hours=float(hours),
                max_actions=int(data['n_bees']*math.ceil(hours*60/data['round_minutes'])),
                model='Qwen3.5-0.8B continued menu LoRA; mixed-state probe experiment')
    if SMOKE: data['max_actions']=min(data['max_actions'],24)
    return Config(**data)

class CapturingPolicy(BeeMenuQwenPolicy):
    def __init__(self,*args,capture=False,**kwargs):
        super().__init__(*args,**kwargs)
        self.capture=bool(capture); self.observations=[]
    def decide_batch(self,observations):
        if self.capture: self.observations.extend(copy.deepcopy(observations))
        return super().decide_batch(observations)

class NormCapturePolicy(CapturingPolicy):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.samples={layer:[] for layer in SELECTED_LAYERS}
    def decide_batch(self,observations):
        handles=[]
        try:
            for layer_i in SELECTED_LAYERS:
                def capture_norm(_module,_inputs,output,layer_i=layer_i):
                    if len(self.samples[layer_i])>=256: return
                    hidden=output[0] if isinstance(output,(tuple,list)) else output
                    values=hidden[:,-1,:].float().norm(dim=-1).detach().cpu().tolist()
                    self.samples[layer_i].extend(values[:256-len(self.samples[layer_i])])
                handles.append(DECODER_LAYERS[layer_i].register_forward_hook(capture_norm))
            return super().decide_batch(observations)
        finally:
            for handle in handles: handle.remove()

class ColumnSteeredPolicy(CapturingPolicy):
    def __init__(self,directions,capture=False):
        super().__init__(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
        self.directions=directions; self.hook_calls=0
        self.label='mixed_probe_column_steered_menu'
    def decide_batch(self,observations):
        handles=[]
        try:
            for layer_i,unit in self.directions.items():
                size=FRACTION*activation_norms[layer_i]/math.sqrt(len(self.directions))
                cached={}
                def steer(_module,_inputs,output,unit=unit,size=size,cached=cached):
                    hidden=output[0] if isinstance(output,(tuple,list)) else output
                    key=(str(hidden.device),hidden.dtype)
                    if key not in cached:
                        cached[key]=torch.as_tensor(unit,device=hidden.device,dtype=hidden.dtype).view(1,1,-1)*size
                    shift=cached[key]
                    assert hidden.shape[-1]==shift.shape[-1]
                    self.hook_calls+=1
                    if POSITION=='response' and hidden.shape[1]>1:
                        altered=hidden.clone(); altered[:,-1:,:]+=shift
                    else:
                        altered=hidden+shift
                    if isinstance(output,tuple): return (altered,*output[1:])
                    if isinstance(output,list): return [altered,*output[1:]]
                    return altered
                handles.append(DECODER_LAYERS[layer_i].register_forward_hook(steer))
            result=super().decide_batch(observations)
            assert self.hook_calls>0
            return result
        finally:
            for handle in handles: handle.remove()

def run_arm(name,seed,hours,directions=None,capture=False,capture_norm=False):
    world=World(config_for(seed,hours))
    if capture_norm:
        policy=NormCapturePolicy(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
    elif directions is None:
        policy=CapturingPolicy(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
    else:
        policy=ColumnSteeredPolicy(directions,capture=capture)
    output_dir=RUN_ROOT/name
    sim=Simulation(world,policy,output_dir=output_dir)
    transcript=sim.run(progress_every=200,decision_batch_size=BATCH_SIZE)
    audit=sim.audit; outcomes=Counter(r['outcome']['status'] for r in audit)
    initial=sim.baseline; final=transcript[-1]['vector']
    immediate=float(sum(np.mean(r['immediate_action_delta']) for r in audit))
    background=float(sum(np.mean(r['background_delta']) for r in audit))
    row={'seed':int(seed),'name':name,'hours':hours,'actions':len(audit),
         'applied':outcomes['applied'],'rejected':outcomes['rejected'],
         'schema_invalid':sum(validate_action(r['action']) is not None for r in audit),
         'initial_vector':initial,'final_vector':final,
         'initial_score':weighted_research_score(initial,[1.0]*14),
         'final_score':weighted_research_score(final,[1.0]*14),
         'immediate_score_change':immediate,'background_score_change':background,
         'hook_calls':getattr(policy,'hook_calls',0),'output_dir':str(output_dir)}
    assert len(audit)==config_for(seed,hours).max_actions
    assert abs(row['initial_score']+immediate+background-row['final_score'])<1e-6
    assert all(len(x['vector'])==14 and all(np.isfinite(x['vector'])) and
               all(0<=v<=1 for v in x['vector']) for x in transcript)
    if directions is not None: assert row['hook_calls']>0
    if capture: assert len(policy.observations)==len(audit)
    print(name,'score',round(row['final_score'],6),'applied',row['applied'],
          'rejected',row['rejected'],flush=True)
    return row,policy

def action_list(row):
    return [json.loads(line)['action'] for line in
            (Path(row['output_dir'])/'actions.jsonl').open()]

def annotate_changes(row,baseline):
    left,right=action_list(row),action_list(baseline)
    assert len(left)==len(right)
    row['changed_actions_vs_baseline']=sum(a!=b for a,b in zip(left,right))
    row['score_difference_vs_baseline']=row['final_score']-baseline['final_score']
    row['vector_difference_vs_baseline']=(np.asarray(row['final_vector'])-
                                          np.asarray(baseline['final_vector'])).tolist()
    return row

In [ ]:
class ReplayCapturePolicy:
    label='saved_action_replay_for_probe_training'
    def __init__(self,world,logged,collect):
        self.world=world;self.logged=logged;self.collect=collect
        self.position=0;self.calls=0;self.observations=[];self.checkpoints=[]
    def decide_batch(self,observations):
        self.observations.extend(copy.deepcopy(observations))
        keep=self.calls%4==0 and (not SMOKE or self.calls==0)
        if keep:
            self.checkpoints.append({'seed':self.world.config.seed,
                'arm':self.collect,'hour':1+int((self.world.minutes-1e-8)//60),
                'world':copy.deepcopy(self.world),'observations':copy.deepcopy(observations)})
        count=len(observations)
        chosen=[row['action'] for row in self.logged[self.position:self.position+count]]
        assert len(chosen)==count
        self.position+=count;self.calls+=1
        return chosen

train_examples=[];training_checkpoints=[];replay_rows=[]
for seed in (90,91):
    for arm in ('baseline','bootstrap_equal'):
        folder=PROBE_ROOT/f'source_{seed}_{arm}'
        metadata=json.loads((folder/'metadata.json').read_text())
        logged=[json.loads(line) for line in (folder/'actions.jsonl').open()]
        saved=[json.loads(line)['vector'] for line in (folder/'vectors.jsonl').open()]
        world=World(Config(**metadata['calibration']['config']))
        policy=ReplayCapturePolicy(world,logged,arm)
        sim=Simulation(world,policy)
        replayed=sim.run(decision_batch_size=BATCH_SIZE)
        assert len(replayed)==len(saved)==len(logged)==400
        error=float(np.max(np.abs(np.asarray([row['vector'] for row in replayed])-saved)))
        assert error==0.0,(seed,arm,error)
        replay_rows.append({'seed':seed,'arm':arm,'actions':len(logged),
                            'max_vector_error':error,'checkpoints':len(policy.checkpoints)})
        training_checkpoints.extend(policy.checkpoints)
        for i,(observation,audit) in enumerate(zip(policy.observations,sim.audit)):
            if SMOKE and i>=24:continue
            train_examples.append({'seed':seed,'arm':arm,'hour':1+i//200,
                'kind':'natural','observation':observation,'action':audit['action'],
                'target':audit['immediate_action_delta']})
pd.DataFrame(replay_rows).to_csv(RUN_ROOT/'source_replay_audit.csv',index=False)
print('Exact source replays',len(replay_rows),'natural training examples',
      len(train_examples),'frozen training batches',len(training_checkpoints))

In [ ]:
def labeled_action(world,bee_id,action):
    before=np.asarray(Observer(world).snapshot(),dtype=np.float64)
    outcome=world.execute(bee_id,action,world.config.round_minutes)
    if outcome['status']!='applied':return None
    after=np.asarray(Observer(world).snapshot(),dtype=np.float64)
    return (after-before).tolist()

def targeted_examples(checkpoints,training):
    examples=[]
    quotas={'feed_brood':8,'groom':4,'heat':4,'clean':4,'build_comb':4}
    for checkpoint in checkpoints:
        world=checkpoint['world'];observations=checkpoint['observations']
        for kind,limit in quotas.items():
            selected=[]
            for observation in observations:
                action=next((a for a in feasible_actions(observation)
                             if a['action']==kind),None)
                if action is not None:selected.append((observation,action))
            for observation,action in selected[:limit]:
                clone=copy.deepcopy(world)
                target=labeled_action(clone,observation['self']['id'],action)
                if target is None:continue
                if kind=='heat' and abs(target[8])<1e-10:continue
                examples.append({'seed':checkpoint['seed'],'arm':checkpoint.get('arm','baseline'),
                    'hour':checkpoint['hour'],'kind':'targeted_'+kind,
                    'observation':observation,'action':action,'target':target})
        if not training:continue
        # Controlled states are training augmentation only. Every label is
        # calculated by actually executing the candidate action in its clone.
        brood_bees=[o for o in observations if o['current_cell']['zone']=='brood'][:4]
        for index,original in enumerate(brood_bees):
            bee_id=original['self']['id'];cell_id=original['self']['cell']
            clone=copy.deepcopy(world)
            new_id=f'counterfactual-larva-{checkpoint["seed"]}-{index}'
            clone.brood[new_id]=Brood(new_id,cell_id,initial=False,hunger=0.8)
            observation=clone.observe(bee_id)
            action=next((a for a in feasible_actions(observation)
                         if a['action']=='feed_brood' and a['target']==new_id),None)
            if action is not None:
                target=labeled_action(clone,bee_id,action)
                if target is not None and target[4]<-1e-10 and abs(target[7])<1e-10:
                    examples.append({'seed':checkpoint['seed'],'arm':checkpoint.get('arm','baseline'),
                        'hour':checkpoint['hour'],'kind':'synthetic_pollen_only',
                        'observation':observation,'action':action,'target':target})
        for original in observations[:4]:
            bee_id=original['self']['id'];clone=copy.deepcopy(world)
            cell=clone.cells[clone.bees[bee_id].cell]
            if cell.zone=='outside' or cell.debris<=1e-5:continue
            cell.pathogen=0.0
            observation=clone.observe(bee_id)
            action=next((a for a in feasible_actions(observation)
                         if a['action']=='clean'),None)
            if action is None:continue
            target=labeled_action(clone,bee_id,action)
            if target is not None and target[10]>1e-10 and abs(target[12])<1e-10:
                examples.append({'seed':checkpoint['seed'],'arm':checkpoint.get('arm','baseline'),
                    'hour':checkpoint['hour'],'kind':'synthetic_hygiene_only',
                    'observation':observation,'action':action,'target':target})
        for original in brood_bees:
            bee_id=original['self']['id'];clone=copy.deepcopy(world)
            clone.cells[clone.bees[bee_id].cell].temperature_c=30.0
            observation=clone.observe(bee_id)
            action=next((a for a in feasible_actions(observation)
                         if a['action']=='heat'),None)
            if action is None:continue
            target=labeled_action(clone,bee_id,action)
            if target is not None and target[8]>1e-10:
                examples.append({'seed':checkpoint['seed'],'arm':checkpoint.get('arm','baseline'),
                    'hour':checkpoint['hour'],'kind':'synthetic_cool_brood_heat',
                    'observation':observation,'action':action,'target':target})
    return examples

augmented=targeted_examples(training_checkpoints,training=True)
train_examples.extend(augmented)
print('Training examples',len(train_examples),'augmentation',
      dict(Counter(row['kind'] for row in augmented)))
if not SMOKE:
    assert set(row['hour'] for row in train_examples)=={1,2}
    assert {'baseline','bootstrap_equal'}<=set(row['arm'] for row in train_examples)
    assert sum(row['kind']=='synthetic_pollen_only' for row in augmented)>=20
    assert sum(row['kind']=='synthetic_hygiene_only' for row in augmented)>=20

In [ ]:
if not SMOKE:
    assert len(train_examples)==TRAINED_REPORT['training_examples']==2623
    assert dict(Counter(row['kind'] for row in train_examples))==           TRAINED_REPORT['training_kind_counts']
    assert all(row['max_vector_error']==0 for row in replay_rows)

def encoded_chat(messages,add_generation_prompt):
    enc=processor.apply_chat_template(messages,tokenize=True,
        add_generation_prompt=add_generation_prompt,return_tensors='pt',return_dict=True)
    return {k:v for k,v in enc.items() if isinstance(v,torch.Tensor)}

pooled={};response_span=[0,0];handles=[]
for layer_i in SELECTED_LAYERS:
    def capture(_module,_inputs,output,layer_i=layer_i):
        hidden=output[0] if isinstance(output,(tuple,list)) else output
        start,end=response_span
        pooled[layer_i]=hidden[0,start:end].float().mean(dim=0).detach().cpu().numpy()
    handles.append(DECODER_LAYERS[layer_i].register_forward_hook(capture))
norm_samples={layer:[] for layer in SELECTED_LAYERS}
started=time.perf_counter()
try:
    for index,row in enumerate(train_examples):
        enriched=dict(row['observation'],feasible_actions=feasible_actions(row['observation']))
        prompt=[{'role':'system','content':MENU_SYSTEM_PROMPT},
                {'role':'user','content':json.dumps(enriched,ensure_ascii=False,
                                                  separators=(',',':'),allow_nan=False)}]
        answer=json.dumps(row['action'],ensure_ascii=False,separators=(',',':'),allow_nan=False)
        p=encoded_chat(prompt,True)
        f=encoded_chat(prompt+[{'role':'assistant','content':answer}],False)
        pids,fids=p['input_ids'][0],f['input_ids'][0]
        assert torch.equal(pids,fids[:len(pids)])
        response_span[:]=[len(pids),len(fids)-1]
        assert response_span[1]>response_span[0] and len(fids)<=4096
        pooled.clear()
        with torch.inference_mode():
            output=model(**{k:v.to(model.device) for k,v in f.items()},use_cache=False)
        del output
        for layer_i in SELECTED_LAYERS:
            norm_samples[layer_i].append(float(np.linalg.norm(pooled[layer_i])))
        if (index+1)%100==0:
            print('Norm reconstruction',index+1,'/',len(train_examples),'minutes',
                  round((time.perf_counter()-started)/60,1),flush=True)
finally:
    for handle in handles:handle.remove()
activation_norms.update({layer:float(np.median(norm_samples[layer]))
                         for layer in SELECTED_LAYERS})
assert all(np.isfinite(activation_norms[layer]) and activation_norms[layer]>0
           for layer in SELECTED_LAYERS)
norm_record={'source_training_examples':len(train_examples),
             'probe_layers':list(SELECTED_LAYERS),
             'activation_norms':activation_norms,
             'method':'median neutral teacher-forced response-feature norm',
             'source_replay_errors':[row['max_vector_error'] for row in replay_rows]}
(RUN_ROOT/'reconstructed_activation_norms.json').write_text(json.dumps(norm_record,indent=2))
print('Reconstructed fixed activation norms',activation_norms)

## 2. Load frozen signed directions and evaluate four paired seeds

The saved sign and weight dictionaries are loaded unchanged. Each normalized
probe column receives its saved sign before combination. The equal and learned
combinations are each renormalized at both selected layers, then injected at
the fixed 0.045 fraction of the reconstructed activation norms. The policy
and simulator use the same 50 bees, 15-minute rounds, batch size 16,
two-hour horizon, and frozen-world batch semantics as the prior experiment.

In [ ]:
def build_combo(weight_by_index):
    weights=np.asarray([weight_by_index.get(j,0.0) for j in ACTIVE],dtype=np.float32)
    assert np.any(weights!=0)
    return {layer:normalized(U[layer]@weights) for layer in SELECTED_LAYERS}

signs={j:int(TRAINED_REPORT['chosen_signs'][VECTOR_NAMES[j]]) for j in ACTIVE}
equal_weights={j:signs[j] for j in ACTIVE}
learned_weights={j:signs[j]*float(TRAINED_REPORT['learned_weights'][VECTOR_NAMES[j]])
                 for j in ACTIVE}
EQUAL_DIR=build_combo(equal_weights)
LEARNED_DIR=build_combo(learned_weights)
assert all(abs(float(np.linalg.norm(direction))-1)<1e-5
           for combo in (EQUAL_DIR,LEARNED_DIR) for direction in combo.values())
print('Frozen signs',signs,'learned weights',learned_weights,
      'strength',FRACTION)

final_rows=[]
for seed in FINAL_SEEDS:
    baseline,_=run_arm(f'holdout_{seed}_baseline',seed,HOURS)
    baseline['arm']='baseline';baseline['changed_actions_vs_baseline']=0
    baseline['score_difference_vs_baseline']=0.0
    baseline['vector_difference_vs_baseline']=[0.0]*14
    final_rows.append(baseline)
    for name,direction in (('new_equal_combination',EQUAL_DIR),
                           ('new_learned_combination',LEARNED_DIR)):
        row,_=run_arm(f'holdout_{seed}_{name}',seed,HOURS,direction)
        row['arm']=name
        final_rows.append(annotate_changes(row,baseline))

summary=pd.DataFrame([{k:v for k,v in row.items() if k not in
                       ('initial_vector','final_vector','vector_difference_vs_baseline')}
                       for row in final_rows])
summary.to_csv(RUN_ROOT/'holdout_run_summary.csv',index=False)
pd.DataFrame([{'seed':row['seed'],'arm':row['arm'],'attribute':VECTOR_NAMES[j],
               'initial':row['initial_vector'][j],'final':row['final_vector'][j],
               'minus_baseline':row['vector_difference_vs_baseline'][j]}
              for row in final_rows for j in range(14)]).to_csv(
                  RUN_ROOT/'holdout_final_vectors.csv',index=False)
print(summary[['seed','arm','final_score','score_difference_vs_baseline',
               'changed_actions_vs_baseline']].to_string(index=False))

## 3. Export every vector step and overlay trajectories

The transcript contains each final arm's timestamp, action, immediate,
background, and total 14-component step change, and post-action state.
The notebook audits the arithmetic and saves three overlaid score
trajectories across the four paired seeds.

In [ ]:
step_rows=[]
for row in final_rows:
    output_dir=Path(row['output_dir'])
    actions=[json.loads(line) for line in (output_dir/'actions.jsonl').open()]
    vectors=[json.loads(line) for line in (output_dir/'vectors.jsonl').open()]
    metadata=json.loads((output_dir/'metadata.json').read_text())
    assert metadata['completed'] and len(actions)==len(vectors)==row['actions']
    start=pd.Timestamp(metadata['initial_timestamp'])
    previous=np.asarray(metadata['initial_vector'],dtype=np.float64)
    for action,vector in zip(actions,vectors):
        immediate=np.asarray(action['immediate_action_delta'],dtype=np.float64)
        background=np.asarray(action['background_delta'],dtype=np.float64)
        current=np.asarray(vector['vector'],dtype=np.float64)
        assert immediate.shape==background.shape==current.shape==(14,)
        assert np.allclose(previous+immediate+background,current,rtol=0,atol=1e-8)
        assert np.all((0<=current)&(current<=1))
        item={'seed':row['seed'],'arm':row['arm'],'event_id':action['event_id'],
              'bee_id':action['bee_id'],'timestamp':action['timestamp'],
              'sim_hour':(pd.Timestamp(action['timestamp'])-start).total_seconds()/3600,
              'action':action['action']['action'],'outcome':action['outcome']['status'],
              'hive_score':float(current.mean())}
        for j,name in enumerate(VECTOR_NAMES):
            item['value_'+name]=float(current[j])
            item['immediate_'+name]=float(immediate[j])
            item['background_'+name]=float(background[j])
            item['total_'+name]=float(immediate[j]+background[j])
        step_rows.append(item);previous=current
    assert np.allclose(previous,row['final_vector'],rtol=0,atol=1e-8)
steps=pd.DataFrame(step_rows)
assert len(steps)==sum(row['actions'] for row in final_rows)
steps.to_csv(RUN_ROOT/'holdout_step_changes.csv.gz',index=False,compression='gzip')

import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(11,5),constrained_layout=True)
colors={'baseline':'#555555','new_equal_combination':'#087e8b',
        'new_learned_combination':'#c17c00'}
for arm,color in colors.items():
    part=steps[steps.arm==arm]
    mean=part.groupby('event_id').agg({'sim_hour':'mean','hive_score':'mean'})
    ax.plot(mean.sim_hour,mean.hive_score,label=arm,color=color,linewidth=1.5)
ax.set(xlabel='Simulated hours',ylabel='Mean 14-attribute hive score',
       title='Frozen retrained-probe steering on seeds 100–103')
ax.legend();ax.grid(alpha=0.2)
fig.savefig(RUN_ROOT/'holdout_mean_score_trajectories.png',dpi=150)
plt.close(fig)

In [ ]:
def bootstrap_interval(values,draws=10000):
    values=np.asarray(values,dtype=np.float64)
    rng=np.random.default_rng(121)
    means=rng.choice(values,size=(draws,len(values)),replace=True).mean(axis=1)
    return np.quantile(means,[0.025,0.975]).tolist()

paired={}
for arm in ('new_equal_combination','new_learned_combination'):
    rows=[row for row in final_rows if row['arm']==arm]
    differences=[row['score_difference_vs_baseline'] for row in rows]
    paired[arm]={'score_differences':differences,
                 'mean_score_difference':float(np.mean(differences)),
                 'bootstrap_95pct_interval':bootstrap_interval(differences),
                 'changed_actions':sum(row['changed_actions_vs_baseline'] for row in rows),
                 'mean_final_vector':np.mean([row['final_vector'] for row in rows],axis=0).tolist()}
report={'model':'continued Qwen3.5-0.8B bee menu LoRA',
        'probe_source':str(TRAINED_ROOT),
        'probe_layers':list(SELECTED_LAYERS),
        'reconstructed_activation_norms':activation_norms,
        'strength':FRACTION,'position':POSITION,
        'signs':dict((VECTOR_NAMES[j],signs[j]) for j in ACTIVE),
        'learned_weights':dict((VECTOR_NAMES[j],abs(learned_weights[j])) for j in ACTIVE),
        'no_probe_retraining_or_refitting':True,
        'seed_history_note':'100–103 were used in earlier experiments, not in retrained-probe fitting or calibration',
        'smoke':SMOKE,'seeds':list(FINAL_SEEDS),'hours':HOURS,
        'final_runs':final_rows,'paired':paired,'step_records':len(steps)}
(RUN_ROOT/'holdout_report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
print('Paired summary',json.dumps({k:{'mean':v['mean_score_difference'],
                                    'interval':v['bootstrap_95pct_interval'],
                                    'changed_actions':v['changed_actions']}
                                   for k,v in paired.items()},indent=2))
print('Artifacts',RUN_ROOT)